In [11]:
import numpy as np
import pandas as pd

In [13]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder


In [19]:
df=pd.read_csv('covid_toy.csv')

In [21]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [27]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [29]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [31]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [37]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

**1.WithOut Column Transformer**

SimpleImputer is a preprocessing tool in machine learning (from scikit-learn) used to fill missing values in a dataset.

When your data contains empty values (NaN), many ML algorithms cannot work directly. SimpleImputer replaces those missing values with a chosen value.

Syntax
from sklearn.impute import SimpleImputer
Common strategies
Strategy	What it does
mean	Replaces missing values with the average
median	Replaces with the middle value
most_frequent	Replaces with the most common value
constant	Replaces with a fixed value
Example 1: Mean strategy
import numpy as np
from sklearn.impute import SimpleImputer

data = [[10], [20], [np.nan], [40]]

imputer = SimpleImputer(strategy='mean')

result = imputer.fit_transform(data)

print(result)

Output:

[[10.]
 [20.]
 [23.33]
 [40.]]

Explanation:

Missing value = NaN
Mean = (10 + 20 + 40) / 3 = 23.33
SimpleImputer inserts 23.33
Example 2: Most frequent (categorical data)
data = [['Red'], ['Blue'], [None], ['Red']]

imputer = SimpleImputer(strategy='most_frequent')

result = imputer.fit_transform(data)

print(result)

Output:

[['Red']
 ['Blue']
 ['Red']
 ['Red']]

Explanation:

"Red" appears most
Missing value becomes "Red"
Methods
fit() → learns replacement value
transform() → replaces missing values
fit_transform() → does both together

It is commonly used before training ML models to clean incomplete datasets.

In [42]:
# 1.adding simple imputer to ferver column

si=SimpleImputer()
x_train_fever=si.fit_transform(x_train[['fever']])

# same with the x_test data
x_test_fever=si.fit_transform(x_test[['fever']])

In [80]:
x_train_fever.shape

(80, 1)

In [48]:
x_test_fever.shape


(20, 1)

In [58]:
# 2. Ordinal Transformer encoding the cough
oe=OrdinalEncoder(categories=[['Mild','Strong']])
x_train_cough=oe.fit_transform(x_train[['cough']])

# Also on train data
x_test_cough=oe.fit_transform(x_test[['cough']])
x_train_cough.shape

(80, 1)

In [60]:
x_train_cough

array([[1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [1.],
       [1.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [1.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [1.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [1.],
       [0.],
       [1.],
       [1.],
       [0.],
       [1.],

In [64]:
# OneHotEncoder on Gender and City columns
ohe=OneHotEncoder(drop='first',sparse_output=False)
x_train_gender_city=ohe.fit_transform(x_train[['gender','city']])

# also the test data
x_test_gender_city=ohe.fit_transform(x_test[['gender','city']])


In [66]:
# Extracting Age 
x_train_age=x_train.drop(columns=['gender','fever','cough','city']).values
# also test data
x_test_age=x_test.drop(columns=['gender','fever','cough','city']).values

In [72]:
x_train_transformed=np.concatenate((x_train_age,x_train_fever,x_train_gender_city,x_train_cough),axis=1)
#also test
x_test_transformed=np.concatenate((x_test_age,x_test_fever,x_test_gender_city,x_test_cough),axis=1)


In [76]:
x_train_transformed.shape

(80, 7)

*  2. WITH Column Transformer *

In [93]:
from sklearn.compose import ColumnTransformer

In [97]:
transformer=ColumnTransformer(transformers=[('tnf1',SimpleImputer(),['fever']),
                                            ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
                                            ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
                                           ],remainder='passthrough')

In [99]:
transformer.fit_transform(x_train).shape

(80, 7)

In [101]:
transformer.fit_transform(x_test).shape

(20, 7)